# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ikramkhan-gif1/FlyRank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My rule

I will prioritize pages for review when they show signs of declining recent visibility and/or have not been updated for a long time.

I will use two observable signals:

1. `days_since_last_update` — a staleness signal linked to FlyRank's refresh-review logic.
2. `impressions_last_30d` compared with `impressions_prev_30d` — a recent visibility-change signal.

The baseline score will be simple and hand-written:

- High staleness (`>= 180` days) = 2 points.
- Moderate staleness (`91–180` days) = 1 point.
- Declining impressions (`<= -20%`) = 2 points.
- Otherwise = 0 decline points.

### Reason codes

- `STALE_AND_DECLINING` — highly stale and declining impressions.
- `STALE` — stale without a strong measurable decline.
- `DECLINING` — declining impressions without significant staleness.
- `MONITOR` — neither signal is strong enough for an action.

The score is intended for decision-support, not as proof that refreshing a page will improve performance.

In [19]:
import os
import pandas as pd

REPO_PATH = "/content/FlyRank-ML-Internship"

if not os.path.exists(REPO_PATH):
    !git clone https://github.com/Ikramkhan-gif1/FlyRank-ML-Internship.git

DATA_PATH = os.path.join(
    REPO_PATH,
    "data",
    "raw",
    "content_refresh_anonymized.csv"
)

OUTPUT_DIR = os.path.join(
    REPO_PATH,
    "work",
    "outputs"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Loaded successfully.")

Dataset shape: (30000, 44)
Loaded successfully.


In [20]:
print([
    c for c in df.columns
    if any(x in c.lower() for x in [
        "stale",
        "impression",
        "position",
        "ctr",
        "refresh",
        "update",
        "volume"
    ])
])

['search_volume', 'impressions_90d', 'days_with_impressions', 'impressions_last_30d', 'impressions_prev_30d', 'days_since_last_update', 'ctr', 'avg_position', 'impression_tier', 'position_tier']


In [21]:
df["impression_change_pct"] = (
    (
        df["impressions_last_30d"]
        - df["impressions_prev_30d"]
    )
    / df["impressions_prev_30d"].replace(0, pd.NA)
) * 100

print(df["impression_change_pct"].describe())

count     26612.0
unique    18501.0
top        -100.0
freq       1395.0
Name: impression_change_pct, dtype: float64


### Signal 1 — Staleness

I will test whether pages with more days since their last update show weaker recent visibility. This is linked to FlyRank's refresh-review logic because staleness is a signal behind refresh decisions.

I will bucket `days_since_last_update` into Low, Medium, and High and compare the observed `impressions_last_30d` across the buckets.

**Verdict: MIXED.**

The observed data shows that highly stale pages have much lower recent impressions, but the relationship is not monotonic because the Medium bucket has higher impressions than the Low bucket. Staleness therefore appears directionally useful for identifying some weak pages, but it is not sufficient by itself.

In [22]:
staleness_bins = [-1, 90, 180, float("inf")]
staleness_labels = ["Low", "Medium", "High"]

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=staleness_bins,
    labels=staleness_labels
)

staleness_check = (
    df.groupby("staleness_bucket", observed=True)
      .agg(
          n=("days_since_last_update", "size"),
          avg_impressions_30d=("impressions_last_30d", "mean"),
          median_impressions_30d=("impressions_last_30d", "median")
      )
      .reset_index()
)

print(staleness_check)

  staleness_bucket      n  avg_impressions_30d  median_impressions_30d
0              Low  20655          1186.065456                    86.0
1           Medium   9171          2001.390906                   305.0
2             High    174           108.183908                     4.0


### Signal 2 — Recent impression change

I will test whether the change from previous-30-day impressions to last-30-day impressions separates pages with different levels of recent visibility.

I will bucket the percentage change into Declining, Stable, Growing, and No_previous_data.

**Verdict: CONFIRMED.**

The observed buckets show meaningful differences in recent visibility. Declining pages have a lower median recent-impression level than Stable pages, supporting recent impression change as a useful directional signal for prioritizing review.

In [23]:
def change_bucket(x):
    if pd.isna(x):
        return "No_previous_data"
    elif x <= -20:
        return "Declining"
    elif x >= 20:
        return "Growing"
    else:
        return "Stable"


df["impression_change_bucket"] = (
    df["impression_change_pct"].apply(change_bucket)
)

change_check = (
    df.groupby("impression_change_bucket", observed=True)
      .agg(
          n=("impression_change_pct", "size"),
          avg_impressions_30d=("impressions_last_30d", "mean"),
          median_impressions_30d=("impressions_last_30d", "median")
      )
      .reset_index()
)

print(change_check)

  impression_change_bucket      n  avg_impressions_30d  median_impressions_30d
0                Declining  16305           939.433671                   128.0
1                  Growing   4414          2161.976439                   226.0
2         No_previous_data   3388           105.896104                     1.0
3                   Stable   5893          2995.512642                   559.0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline rule

I prioritize pages for review when their recent visibility is declining and/or they have not been updated for a long time.

The baseline score is deliberately simple and hand-written:

- High staleness (`>= 180` days) adds 2 points.
- Moderate staleness (`91–180` days) adds 1 point.
- Declining impressions (`<= -20%`) adds 2 points.
- Stable or growing impressions add no decline points.
- Pages with no previous-period impressions receive no decline points because the change cannot be measured reliably.

### Actions

- Score 4 → `REFRESH_NOW`
- Score 2–3 → `REVIEW`
- Score 0–1 → `MONITOR`

### Reason codes

- `STALE_AND_DECLINING` — high staleness and declining impressions.
- `STALE` — high or moderate staleness without a measurable decline.
- `DECLINING` — declining impressions without high/moderate staleness.
- `MONITOR` — neither signal is strong enough for an action.

In [24]:
# Calculate staleness score
df["stale_score"] = 0

df.loc[
    df["days_since_last_update"].between(91, 180),
    "stale_score"
] = 1

df.loc[
    df["days_since_last_update"] > 180,
    "stale_score"
] = 2


# Calculate decline score
df["decline_score"] = (
    (df["impression_change_pct"] <= -20)
    .fillna(False)
    .astype(int)
    * 2
)


# Calculate total baseline score
df["baseline_score"] = (
    df["stale_score"] + df["decline_score"]
)


# Convert score to action
def action_label(score):
    if score >= 4:
        return "REFRESH_NOW"
    elif score >= 2:
        return "REVIEW"
    else:
        return "MONITOR"


df["action"] = df["baseline_score"].apply(action_label)


# Create reason code
def reason_code(row):
    if row["stale_score"] == 2 and row["decline_score"] == 2:
        return "STALE_AND_DECLINING"
    elif row["decline_score"] == 2:
        return "DECLINING"
    elif row["stale_score"] > 0:
        return "STALE"
    else:
        return "MONITOR"


df["reason_code"] = df.apply(reason_code, axis=1)


print("Baseline score created successfully.")

print(
    df[
        ["baseline_score", "reason_code", "action"]
    ].value_counts().sort_index()
)

Baseline score created successfully.
baseline_score  reason_code          action     
0               MONITOR              MONITOR        10045
1               STALE                MONITOR         3559
2               DECLINING            REVIEW         10610
                STALE                REVIEW            91
3               DECLINING            REVIEW          5612
4               STALE_AND_DECLINING  REFRESH_NOW       83
Name: count, dtype: int64


In [25]:
# Build ranked queue
queue = df.copy()

queue = queue.sort_values(
    by=["baseline_score", "impressions_last_30d"],
    ascending=[False, False]
).reset_index(drop=True)

# Add rank
queue["rank"] = range(1, len(queue) + 1)

# Keep only required queue columns
queue = queue[
    [
        "rank",
        "baseline_score",
        "action",
        "reason_code",
        "days_since_last_update",
        "impressions_last_30d",
        "impressions_prev_30d",
        "impression_change_pct"
    ]
]

# Save required CSV
output_path = os.path.join(
    OUTPUT_DIR,
    "baseline_action_score.csv"
)

queue.to_csv(output_path, index=False)

print("Queue shape:", queue.shape)
print("Saved to:", output_path)

print("\nTop 20:")
display(queue.head(20))

Queue shape: (30000, 8)
Saved to: /content/FlyRank-ML-Internship/work/outputs/baseline_action_score.csv

Top 20:


,rank,baseline_score,action,reason_code,days_since_last_update,impressions_last_30d,impressions_prev_30d,impression_change_pct
0,1,4,REFRESH_NOW,STALE_AND_DECLINING,194,3864,26791,-85.577246
1,2,4,REFRESH_NOW,STALE_AND_DECLINING,194,3778,20472,-81.545526
2,3,4,REFRESH_NOW,STALE_AND_DECLINING,194,2305,9101,-74.673113
3,4,4,REFRESH_NOW,STALE_AND_DECLINING,193,2246,4662,-51.823252
4,5,4,REFRESH_NOW,STALE_AND_DECLINING,193,803,2160,-62.824074
5,6,4,REFRESH_NOW,STALE_AND_DECLINING,194,746,1562,-52.240717
6,7,4,REFRESH_NOW,STALE_AND_DECLINING,194,554,1831,-69.74331
7,8,4,REFRESH_NOW,STALE_AND_DECLINING,194,548,2142,-74.416433
8,9,4,REFRESH_NOW,STALE_AND_DECLINING,193,381,702,-45.726496
9,10,4,REFRESH_NOW,STALE_AND_DECLINING,194,292,2670,-89.06367


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

The top 20 rows are the highest-ranked items produced by the hand-written baseline rule.

For each row, I record the action, reason code, a confidence note about how strongly the row matches the rule, and what could make the recommendation wrong.

The confidence note describes confidence in the rule match, not certainty that the recommended action will improve performance.

Potential reasons a pick could be wrong include seasonality, changing search demand, SERP changes, technical issues, or very low recent impression volume.

In [26]:
# Select the top 20 ranked rows
top20 = queue.head(20).copy()


# Add confidence note
def confidence_note(row):
    if row["impressions_last_30d"] >= 500:
        return "Strong rule match with meaningful recent visibility."
    elif row["impressions_last_30d"] >= 100:
        return "Good rule match, but lower recent visibility."
    else:
        return "Rule match is strong, but very low recent visibility weakens the pick."


# Add explanation of what could make the pick wrong
def wrong_reason(row):
    if row["impressions_last_30d"] < 100:
        return "Very low traffic means the percentage decline may exaggerate the practical opportunity."
    elif row["impressions_last_30d"] < 500:
        return "Moderate traffic; decline may reflect seasonality or changing search demand."
    else:
        return "High traffic does not prove refreshing will recover performance; SERP or demand changes may be responsible."


top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    wrong_reason,
    axis=1
)


print("Top-20 review generated successfully.")

display(
    top20[
        [
            "rank",
            "action",
            "reason_code",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)


Top-20 review generated successfully.


,rank,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,REFRESH_NOW,STALE_AND_DECLINING,Strong rule match with meaningful recent visib...,High traffic does not prove refreshing will re...
1,2,REFRESH_NOW,STALE_AND_DECLINING,Strong rule match with meaningful recent visib...,High traffic does not prove refreshing will re...
2,3,REFRESH_NOW,STALE_AND_DECLINING,Strong rule match with meaningful recent visib...,High traffic does not prove refreshing will re...
3,4,REFRESH_NOW,STALE_AND_DECLINING,Strong rule match with meaningful recent visib...,High traffic does not prove refreshing will re...
4,5,REFRESH_NOW,STALE_AND_DECLINING,Strong rule match with meaningful recent visib...,High traffic does not prove refreshing will re...
5,6,REFRESH_NOW,STALE_AND_DECLINING,Strong rule match with meaningful recent visib...,High traffic does not prove refreshing will re...
6,7,REFRESH_NOW,STALE_AND_DECLINING,Strong rule match with meaningful recent visib...,High traffic does not prove refreshing will re...
7,8,REFRESH_NOW,STALE_AND_DECLINING,Strong rule match with meaningful recent visib...,High traffic does not prove refreshing will re...
8,9,REFRESH_NOW,STALE_AND_DECLINING,"Good rule match, but lower recent visibility.",Moderate traffic; decline may reflect seasonal...
9,10,REFRESH_NOW,STALE_AND_DECLINING,"Good rule match, but lower recent visibility.",Moderate traffic; decline may reflect seasonal...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

Some top-ranked pages may be weak practical picks even when they satisfy the rule. In particular, very low recent impression volume can make percentage declines unstable and can reduce the practical value of a refresh recommendation.

I will flag low-volume rows in the top 20 for manual review rather than treating the baseline action as a guaranteed recommendation.

### Leakage check

The baseline uses only current observable signals:

- `days_since_last_update`
- `impressions_last_30d`
- `impressions_prev_30d`

The baseline does not use the ML target or label, future-window metrics, product flags, client names, URLs, or private queries.

In [27]:
# Identify potentially weak top-20 picks
weak_picks = queue.head(20).copy()

weak_picks = weak_picks[
    weak_picks["impressions_last_30d"] < 100
]

print("Weak top-20 picks (recent impressions < 100):")

display(
    weak_picks[
        [
            "rank",
            "baseline_score",
            "action",
            "reason_code",
            "days_since_last_update",
            "impressions_last_30d",
            "impressions_prev_30d",
            "impression_change_pct"
        ]
    ]
)


# Leakage check
print("\nLeakage check:")

used_columns = [
    "days_since_last_update",
    "impressions_last_30d",
    "impressions_prev_30d"
]

print("Baseline input columns:", used_columns)
print("Target/label used: NO")
print("Future-window inputs used: NO")
print("Product flags used: NO")
print("Client names/URLs/private queries used: NO")

Weak top-20 picks (recent impressions < 100):


,rank,baseline_score,action,reason_code,days_since_last_update,impressions_last_30d,impressions_prev_30d,impression_change_pct
16,17,4,REFRESH_NOW,STALE_AND_DECLINING,301,97,125,-22.4
17,18,4,REFRESH_NOW,STALE_AND_DECLINING,211,89,113,-21.238938
18,19,4,REFRESH_NOW,STALE_AND_DECLINING,183,46,63,-26.984127
19,20,4,REFRESH_NOW,STALE_AND_DECLINING,183,38,60,-36.666667



Leakage check:
Baseline input columns: ['days_since_last_update', 'impressions_last_30d', 'impressions_prev_30d']
Target/label used: NO
Future-window inputs used: NO
Product flags used: NO
Client names/URLs/private queries used: NO


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-check

- [x] Every section is filled with Markdown reasoning and supporting code.
- [x] The notebook runs top to bottom with no errors.
- [x] No client names, URLs, or private queries are included in the analysis.
- [x] Claims use careful language such as observed, measured, directional, and decision-support.
- [x] The baseline does not use the ML target/label or future-window inputs.
- [x] Required output CSV is generated by the notebook.
- [x] Notebook has been committed to the repository under `work/notebooks/`.